# LLMによる情報抽出

LLMは、感情分析や情報抽出などのタスクに適しています。

このNotebookでは、LLMを使用して請求文章を分析し、書き手の心理状態や事故の場所と日時を特定します

### 必要なライブラリとインポート

Labの指示に従って適切なワークベンチイメージを選択して起動した場合、必要なすべてのライブラリがすでにインストールされているはずです。もしインストールされていない場合は、次のセルの最初の行のコメントを外して正しいパッケージをすべてインストールしてください。その後、必要なライブラリをインポートします。

In [ ]:
# !pip install --no-cache-dir --no-dependencies --disable-pip-version-check -r requirements.txt # Uncomment only if you have not selected the right workbench image

import json
import os
from os import listdir
from os.path import isfile, join

from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain.prompts import PromptTemplate
from langchain_community.llms import VLLMOpenAI

### Langchainパイプライン

Langchainを使用して、パイプラインを定義します。

In [ ]:
# LLM推論APIのURL
inference_server_url = "http://llama-3-1-swallow-8b-instruct-v0-3-predictor.ic-shared-llm.svc.cluster.local:8080"

# LLMの定義
llm = VLLMOpenAI(
    openai_api_key="EMPTY",
    openai_api_base= f"{inference_server_url}/v1",
    model_name="llama-3-1-swallow-8b-instruct-v0-3",
    top_p=0.92,
    temperature=0.01,
    max_tokens=512,
    presence_penalty=1.03,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

以下は、タスクに対してフォーマットされた**テンプレート**です。

In [ ]:
template="""<|begin_of_text|><|start_header_id|>system<|end_header_id|>


あなたは、親切で、礼儀正しく、正直なアシスタントです。
常に気配りと尊重をもって接し、真摯にサポートします。できる限り有用な返答を提供しますが、安全を確保します。
有害で、倫理に反する、偏見のある、または否定的な内容は避けます。返答が公正でポジティブなものであることを確認します。<|eot_id|><|start_header_id|>user<|end_header_id|>

与えられた文章の内容をもとに、与えられた質問に答えてください。

### 文章:
{text}

### 質問:
{query}

### 回答:<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

prompt = PromptTemplate(input_variables=["text", "query"], template=template)

モデルにクエリを投げるために使用する**会話**オブジェクトを作成します。

In [ ]:
conversation = prompt | llm

モデルにクエリする準備が整いました。

`claims`フォルダーには、受信される可能性のある請求文章の例がJSONファイルで保存されています。これらのファイルを読み込んで表示し、その後にLLMが行った分析を表示します。

In [ ]:
# 請求文章の読み取り
claims_path = 'claims'
onlyfiles = [f for f in listdir(claims_path) if isfile(join(claims_path, f))]

claims = {}

for filename in onlyfiles:
    with open(os.path.join(claims_path, filename), 'r') as file:
        data = json.load(file)
    claims[filename] = data

In [ ]:
for filename in onlyfiles:
    print(f"***************************")
    print(f"* 請求: {filename}")
    print(f"***************************")
    print("元の文章:")
    print("-----------------")
    print(f"件名: {claims[filename]['subject']}\n内容:\n{claims[filename]['content']}\n\n")
    print('分析:')
    print("--------")
    text_input = f"件名: {claims[filename]['subject']}\n内容:\n{claims[filename]['content']}"
    sentiment_query = "この請求の文章から読み取れる感情はどのようなものですか？「肯定的」、「否定的」、「どちらでもない」から1つだけ選んで答え、その理由もあわせて説明してください。"
    location_query = "この請求に関連する出来事はどこで起こりましたか？出来事の発生した場所について、市区町村や通りの名前などを含めて答えて下さい。"
    time_query = "この請求に関連する出来事はいつ起こりましたか？日付と、時刻あるいは時間帯を一言で答えて下さい。"
    print(f"- 送信者の感情: ")
    conversation.invoke(input={"text": text_input, "query": sentiment_query})
    print("\n- 発生場所: ")
    conversation.invoke(input={"text": text_input, "query": location_query})
    print("\n- 発生日時: ")
    conversation.invoke(input={"text": text_input, "query": time_query})
    print("\n\n                          ----====----\n")